In [0]:
%pip install /dbfs/FileStore/shared_uploads/dmat/pmgoogle-1.4-py3-none-any.whl
%pip install  /dbfs/FileStore/shared_uploads/dmat/dmat-0.1-py3-none-any.whl

dbutils.widgets.text(
    "report_run_date",      # Widget name
    "",                     # Default value
    "Report Run Date"       # Optional label
)

In [0]:
from dmat.helpers import *
from dmat import spark_helpers as dsh

from dateutil.relativedelta import relativedelta
import datetime

spark.conf.set("spark.sql.shuffle.partitions","1024")
spark.conf.set("spark.sql.sources.partitionOverwriteMode","dynamic")
#sc.hadoopConfiguration.set("mapreduce.fileoutputcommitter.marksuccessfuljobs", "false")

##############
## Imports & Inits
import pyspark
from pyspark import *
from pyspark.sql import *
from dateutil import tz
from dateutil.relativedelta import relativedelta
import datetime

from pmgoogle.sheets import GoogleSheet
import numpy as np
import pandas as pd
import time
import json
import pyspark.sql.functions as F

# get service account creds
#athena-master@athena-284716.iam.gserviceaccount.com
#service_account = json.loads(dbutils.secrets.get(scope="data_pa-dmat_dmat-secrets-scope", key="athena_sa"))['general']  

sc = spark._sc
helpers = sc._jvm.com.poshmark.spark.helpers
Config = helpers.Config
Redshift = helpers.Redshift
SaveMode = sc._jvm.org.apache.spark.sql.SaveMode
SaveMode_Append = SaveMode.Append
SaveMode_Overwrite = SaveMode.Overwrite
SaveMode_Ignore = SaveMode.Ignore
SaveMode_Error = SaveMode.ErrorIfExists

## Load default configs for helper libraries and register UDFs
Config.reloadConfigs(sc._jsc.sc())
Config.registerUDFs()


# Returns the current UTC time
def getUTC():
    return datetime.datetime.utcnow().replace(tzinfo=tz.gettz('UTC'))

# Returns the current time in 'US/Pacific'
def getPacificTime(date=False, month=False):
    now = getUTC().astimezone(tz.gettz('US/Pacific'))
    if date:
        now = now.date()
    if month:
        now = now.replace(day=1)

    return now



dsh.init(spark)
Redshift = sc._jvm.com.poshmark.spark.helpers.Redshift

def get_widget(widget_name, default="", _type=str):
    try:
        tmp =  _type(dbutils.widgets.get(widget_name))
        if tmp:
            return tmp
        else:
            return default
    except:
        return default

run_now = get_widget('run_now',default='false')
if (run_now.lower() == 'true'):
    force = True
else:
    force = False

send_day = getPacificTime().isoweekday()
start_of_month = True if getPacificTime().day == 1 else False

## Get today & yesterday
local_date = getPacificTime() 
today = local_date.strftime('%Y-%m-%d')
yesterday = (local_date - datetime.timedelta(days=1)).strftime('%Y-%m-%d')

##############
## get a value from a widget
report_run_date = dbutils.widgets.get("report_run_date")
process_end_date = get_widget('report_run_date', default=yesterday)
## last_90_days = (datetime.datetime.strptime(process_end_date, "%Y-%m-%d") - datetime.timedelta(days=90)).strftime('%Y-%m-%d') 
last_90_days = (datetime.datetime.strptime(process_end_date, "%Y-%m-%d") - datetime.timedelta(days=10)).strftime('%Y-%m-%d')
## last_90_days = (datetime.datetime.strptime(process_end_date, "%Y-%m-%d") - datetime.timedelta(days=2)).strftime('%Y-%m-%d')
process_start_date = last_90_days

## process_date = get_widget("process_date",default=yesterday)
## process_end_date = get_widget("process_end_date",default=yesterday)

# print(f"process_date: {process_date}, process_end_date: {process_end_date}")

# stage_s3_path = "s3://data-tmp-poshmark-production/dmat/project_highway/top_stats/v2/"

def to_jvm_list(py_list):
    l = len(py_list)
    jvm_list = sc._gateway.new_array(sc._jvm.java.lang.String, l)
    for i in range(l):
        jvm_list[i] = py_list[i]
    return jvm_list
print(process_start_date, process_end_date)


In [0]:
#s3_path = "s3://data-tmp-poshmark-production/qiong/for_you_feed/visual_discovery/visual_discovery_feed_content_lvl2/"
s3_path = "s3://analytics-scratch-reviewed-poshmark-production/qiong/for_you_feed/visual_discovery/visual_discovery_feed_content_lvl2/"
df = spark.read.format("parquet").load(s3_path)
df.createOrReplaceTempView("visual_discovery_feed_content_lvl2")

In [0]:
%sql 
select min(event_date), max(event_date) from visual_discovery_feed_content_lvl2 


In [0]:
%sql
create or replace temporary view visual_discovery_feed_creator_base
as 
select 
event_date, 
unit_position,
story_type, 
user_id, 
home_domain, 
user_gender, 
unit_name,
collection_name,
aesthetic_name,
department_name, 
collection_id, 
b.ttl as creator_id, 
b.pnm as pnm, 
CASE 
    WHEN extended_signup_segments_v3 IN 
     ('042', '043', '059', '061', '063', '065', '069', '070', '078', '079', '089', '100', '107', '109', '117', '126') then 'Test'
    WHEN extended_signup_segments_v3 IN ('026', '044', '046', '056', '066', '067', '072', '076', '084', '092', '095', '101', '105', '112', '118', '123') then 'Control'
    ELSE 'Other' 
  END AS exp_grp, 
content_position, 
new_content_position, 
sub_unit_type, 
app_type,

count(case when verb = 'observe' then user_id else null end) as client_impressions,
count(case when verb = 'click' then user_id else null end) as clicks,
count(distinct case when verb = 'observe' then user_id else null end) as client_impressions_users,
count(distinct case when verb = 'click' then user_id else null end) as clicks_users

from visual_discovery_feed_content_lvl2 a
left join s3_raw_mongo.looks b on a.collection_id = b. _id
where 1=1 --event_date between '2025-11-11' and '2025-11-17'
group by 1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18

In [0]:
%sql 
select creator_id, 
 count(distinct collection_id),
 sum(client_impressions) as client_impressions, 
 sum(clicks) as clicks, sum(client_impressions_users) as client_impressions_users,
  sum(clicks_users) as clicks_users
from visual_discovery_feed_creator_base
  group by 1
  order by 4 desc

## Order Attribution 
### 1) find all the usersa on shop the look page to isolate the search events
[ql_visual_discovery_look_page](https://poshmark-prod.cloud.databricks.com/editor/notebooks/4224641565907605?o=3891659053752709)

In [0]:
#bucket = "s3://data-tmp-poshmark-production" ## 30 day bucket
bucket = "s3://analytics-scratch-reviewed-poshmark-production" 
team_name = "product_analytics"
project_name = "feed/visual_discovery/look"
data_name = "lookDF"

output_path = f"{bucket}/{team_name}/{project_name}/{data_name}/"
look_sdf = spark.read.parquet(output_path)
look_sdf.createOrReplaceTempView("ql_visual_discovery_look_page")

In [0]:
searchDF = spark.sql(f""" 
with base as (select 
event_date,
at event_time, 
id event_id,
 verb,
 properties.location, 
 using.app_type, 
 on.name, 
actor.id user_id, 
get_json_object(properties.tracking_meta, '$.provider_id') provider_id, 
get_json_object(properties.tracking_meta, '$.collection_id') collection_id,
r.properties.unit_position as unit_position,
case when r.using.app_type = 'web' then r.properties.content_position - 1 else r.properties.content_position end as new_content_position,
r.properties.content_position as content_position,
r.properties.listing_id, 
lead(r.direct_object.page_group_id, 1) OVER (PARTITION BY r.actor.id ORDER BY at) AS next_search_page_group_id
from  raw_events r
where event_date between '{process_start_date}' and '{process_end_date}'
and verb in ('click','search')
and ((on.name in ('look') -- this will be look only web is fixing bug
and verb='click' and properties.location='footer')  or (verb='search' and properties.cause='listing_search'))
)
select * from base 
where verb = 'click' and next_search_page_group_id is not null
""")



s3_path_old = "s3://data-tmp-poshmark-production/product_analytics/feed/visual_discovery/search/searchDF/"
s3_path = "s3://analytics-scratch-reviewed-poshmark-production/product_analytics/feed/visual_discovery/search/searchDF/"
df = spark.read.format("parquet").load(s3_path_old)
df.repartition("event_date").write.partitionBy("event_date").format("parquet").mode("Overwrite").save(s3_path)


In [0]:
## write base table to path, this will serve as a roll-up to weekly and monthly to reduce the processing time

bucket = "s3://analytics-scratch-reviewed-poshmark-production" ## 1yr bucket
##bucket = "s3://data-tmp-poshmark-production" ## 30 day bucket
team_name = "product_analytics"
project_name = "feed/visual_discovery/search"
data_name = "searchDF"

output_path = f"{bucket}/{team_name}/{project_name}/{data_name}/"

## ## temp change to static from dynamic
##spark.conf.set("spark.sql.sources.partitionOverwriteMode","static")

searchDF.repartition("event_date").write.partitionBy("event_date").format("parquet").mode("Overwrite").save(output_path)
## base_orderitem.write.partitionBy("booked_date").format("parquet").mode("Overwrite").save(output_path)
## base_orderitem_df.write.format("parquet").mode("Overwrite").save(output_path)

## ## revert back to dynamic from static
## spark.conf.set("spark.sql.sources.partitionOverwriteMode","dynamic")

look_sdf = spark.read.parquet(output_path)
look_sdf.createOrReplaceTempView("ql_vs_footer_search")

##print(process_start_date)
##print(process_end_date)
print(output_path)

In [0]:
##bucket = "s3://data-tmp-poshmark-production" ## 30 day bucket
bucket = "s3://analytics-scratch-reviewed-poshmark-production" 
team_name = "product_analytics"
project_name = "feed/visual_discovery/search"
data_name = "searchDF"

output_path = f"{bucket}/{team_name}/{project_name}/{data_name}/"
look_sdf = spark.read.parquet(output_path)
look_sdf.createOrReplaceTempView("ql_vs_footer_search")


In [0]:

listingDF =  spark.sql(f""" 
    
    with search_clicks as (
select 
event_date,
at as event_time, 
r.actor.id as user_id, 
id as event_id, 
f_pm_epoch_to_pacific(r.at) as event_at,
r.verb,
r.on.page_group_id as page_group_id,
r.direct_object.name as dob_name,
r.on.name as on_name,
r.properties.story_type as story_type,
r.properties.unit_position as unit_position,
case when r.using.app_type = 'web' then r.properties.content_position - 1 else r.properties.content_position end as new_content_position,
r.properties.content_position as content_position,
r.properties.content_type as content_type,
r.properties.location as location, 
r.properties.listing_id as listing_id

from  raw_events r 
where event_date between '{process_start_date}' and '{process_end_date}'
--'{process_start_date}' and '{process_end_date}'
and on.name = 'search_results' 
and verb = 'click'
and r.on.page_group_id in (select distinct next_search_page_group_id as page_group_id
 from  ql_vs_footer_search
 where event_date between '{process_start_date}' and '{process_end_date}' 
 --'{process_start_date}' and '{process_end_date}'
 )
)

select 
a.event_date, 
a.name, 
a.collection_id, 
a.next_search_page_group_id as page_group_id,
a.user_id,
a.unit_position, 
a.location, 
a.new_content_position, 
a.content_position, 
a.app_type, 
b.listing_id, 
b.content_type, 
b.on_name as search_name, 
b.verb as search_verb,
b.location as search_location, 
b.unit_position as search_unit_position, 
b.content_position as search_content_position, 
b.new_content_position as search_new_content_postion
from ql_vs_footer_search a
left join search_clicks b on a.next_search_page_group_id = b.page_group_id and a.user_id = b.user_id
where a.event_date between'{process_start_date}' and '{process_end_date}'
""")


s3_path_old = "s3://data-tmp-poshmark-production/product_analytics/feed/visual_discovery/search/listingDF/"
s3_path = "s3://analytics-scratch-reviewed-poshmark-production/product_analytics/feed/visual_discovery/search/listingDF/"
df = spark.read.format("parquet").load(s3_path_old)
df.repartition("event_date").write.partitionBy("event_date").format("parquet").mode("Overwrite").save(s3_path)


In [0]:
## write base table to path, this will serve as a roll-up to weekly and monthly to reduce the processing time

bucket = "s3://analytics-scratch-reviewed-poshmark-production" ## 1yr bucket
##bucket = "s3://data-tmp-poshmark-production" ## 30 day bucket
team_name = "product_analytics"
project_name = "feed/visual_discovery/search"
data_name = "listingDF"

output_path = f"{bucket}/{team_name}/{project_name}/{data_name}/"

## ## temp change to static from dynamic
##spark.conf.set("spark.sql.sources.partitionOverwriteMode","static")

listingDF.repartition("event_date").write.partitionBy("event_date").format("parquet").mode("Overwrite").save(output_path)
## base_orderitem.write.partitionBy("booked_date").format("parquet").mode("Overwrite").save(output_path)
## base_orderitem_df.write.format("parquet").mode("Overwrite").save(output_path)

## ## revert back to dynamic from static
## spark.conf.set("spark.sql.sources.partitionOverwriteMode","dynamic")

look_sdf = spark.read.parquet(output_path)
look_sdf.createOrReplaceTempView("ql_vs_footer_search_listings")

##print(process_start_date)
##print(process_end_date)
print(output_path)

In [0]:
## bucket = "s3://data-tmp-poshmark-production" ## 30 day bucket
bucket = "s3://analytics-scratch-reviewed-poshmark-production" ## 1yr bucket
team_name = "product_analytics"
project_name = "feed/visual_discovery/search"
data_name = "listingDF"

output_path = f"{bucket}/{team_name}/{project_name}/{data_name}/"
look_sdf = spark.read.parquet(output_path)
look_sdf.createOrReplaceTempView("ql_vs_footer_search_listings")


Notes: position starting from 0; 

### step 2 #combine footer click and look click listings

#### App Only First for content and All for search query -- web has a lot of issues ; 

In [0]:
clickappDF  = spark.sql(f"""
with footer as (select 
a.event_date, 
a.name as on_name, 
a.collection_id, 
a.page_group_id, 
a.user_id, 
a.unit_position, 
a.location, 
a.new_content_position, 
a.content_position, 
a.app_type, 
a.listing_id, 
a.search_name, 
a.search_verb, 
a.search_unit_position, 
b.user_query, 
'search' as click_source
from ql_vs_footer_search_listings a 
left join (select distinct user_id, search_page_group_id, user_query from search_impressions_interactions 
    where event_date between '{process_start_date}' and '{process_end_date}' --'2026-01-01' and '2026-02-19' 
    
    and search_page_group_id in (select distinct page_group_id from ql_vs_footer_search_listings
    )
    ) b on b.search_page_group_id = a.page_group_id and a.user_id = b.user_id
where listing_id is not null --and app_type <>'web'
and event_date between '{process_start_date}' and '{process_end_date}'
--'2026-01-01' and '2026-02-19'

) 
,content as (
select 
a.event_date, 
a.on_name, 
a.collection_id, 
'#' as page_group_id, 
a.user_id, 
a.unit_position, 
a.location, 
a.new_content_position, 
a.content_position, 
a.app_type, 
a.listing_id, 
'#' as search_name, 
'#' as search_verb, 
'#' as search_unit_position, 
'#' as user_query, 
'look' as click_source
from ql_visual_discovery_look_page a
where verb = 'click' and location = 'content'
and listing_id is not null --and app_type <>'web'
and event_date between '{process_start_date}' and '{process_end_date}' 
--'2026-01-01' and '2026-02-19' 
) 
select * from content 
union all 
select * from footer 

""")



s3_path_old = "s3://data-tmp-poshmark-production/product_analytics/feed/visual_discovery/search/clickappDF/"
s3_path = "s3://analytics-scratch-reviewed-poshmark-production/product_analytics/feed/visual_discovery/search/clickappDF/"
df = spark.read.format("parquet").load(s3_path_old)
df.repartition("event_date").write.partitionBy("event_date").format("parquet").mode("Overwrite").save(s3_path)


In [0]:

## write base table to path, this will serve as a roll-up to weekly and monthly to reduce the processing time

bucket = "s3://analytics-scratch-reviewed-poshmark-production" ## 1yr bucket
## bucket = "s3://data-tmp-poshmark-production" ## 30 day bucket
team_name = "product_analytics"
project_name = "feed/visual_discovery/search"
data_name = "clickappDF"

output_path = f"{bucket}/{team_name}/{project_name}/{data_name}/"

## ## temp change to static from dynamic
##spark.conf.set("spark.sql.sources.partitionOverwriteMode","static")

clickappDF.repartition("event_date").write.partitionBy("event_date").format("parquet").mode("Overwrite").save(output_path)
## base_orderitem.write.partitionBy("booked_date").format("parquet").mode("Overwrite").save(output_path)
## base_orderitem_df.write.format("parquet").mode("Overwrite").save(output_path)

## ## revert back to dynamic from static
## spark.conf.set("spark.sql.sources.partitionOverwriteMode","dynamic")

look_sdf = spark.read.parquet(output_path)
look_sdf.createOrReplaceTempView("ql_vs_click_app")

##print(process_start_date)
##print(process_end_date)
print(output_path)



In [0]:
bucket = "s3://analytics-scratch-reviewed-poshmark-production" ## 1yr bucket
##bucket = "s3://data-tmp-poshmark-production" ## 30 day bucket
team_name = "product_analytics"
project_name = "feed/visual_discovery/search"
data_name = "clickappDF"

output_path = f"{bucket}/{team_name}/{project_name}/{data_name}/"
look_sdf = spark.read.parquet(output_path)
look_sdf.createOrReplaceTempView("ql_vs_click_app")

##print(process_start_date)
##print(process_end_date)
print(output_path)

## Same Day Order

### Order Attribution 
#### Web
- App: ql_vs_click_app_ordr
- web: Content Click need some work; clicks missing listing Details; 

In [0]:

webviewDF = spark.sql(f"""
select 
  r.event_date, 
  r.on.name as on_name, 
  REGEXP_EXTRACT(r.referred_by.url, '/look/([^?]+)')AS collection_id,
  '#' as page_group_id, 
  r.actor.id as user_id, 
  '#' as unit_position, 
  '#' as location, 
  '#' as new_content_position, 
  '#' as content_position, 
  r.using.app_type, 
  r.properties.listing_id as listing_id, 
  '#' as search_name, 
  '#' as search_verb, 
  '#' as search_unit_position, 
  '#' as user_query, 
 
  'look' as click_source,
  /** extra for view logging**/
  REGEXP_EXTRACT(r.referred_by.url, '/(look)/') as src_,
  r.referred_by.url, 
  r.verb,
  r.direct_object.name

from raw_events r  
where verb = 'view'  
and direct_object.name = 'listing_details'
and event_date between '{process_start_date}' and '{process_end_date}'
and using.app_type = 'web'
and REGEXP_EXTRACT(referred_by.url, '/(look)/')  = 'look'

"""
)


s3_path_old = "s3://data-tmp-poshmark-production/product_analytics/feed/visual_discovery/search/webviewDF/"
s3_path = "s3://analytics-scratch-reviewed-poshmark-production/product_analytics/feed/visual_discovery/search/webviewDF/"
df = spark.read.format("parquet").load(s3_path_old)
df.repartition("event_date").write.partitionBy("event_date").format("parquet").mode("Overwrite").save(s3_path)


In [0]:

## write base table to path, this will serve as a roll-up to weekly and monthly to reduce the processing time

bucket = "s3://analytics-scratch-reviewed-poshmark-production" ## 1yr bucket
##bucket = "s3://data-tmp-poshmark-production" ## 30 day bucket
team_name = "product_analytics"
project_name = "feed/visual_discovery/search"
data_name = "webviewDF"

output_path = f"{bucket}/{team_name}/{project_name}/{data_name}/"

## ## temp change to static from dynamic
##spark.conf.set("spark.sql.sources.partitionOverwriteMode","static")

webviewDF.repartition("event_date").write.partitionBy("event_date").format("parquet").mode("Overwrite").save(output_path)
## base_orderitem.write.partitionBy("booked_date").format("parquet").mode("Overwrite").save(output_path)
## base_orderitem_df.write.format("parquet").mode("Overwrite").save(output_path)

# ## revert back to dynamic from static
## spark.conf.set("spark.sql.sources.partitionOverwriteMode","dynamic")

look_sdf = spark.read.parquet(output_path)
look_sdf.createOrReplaceTempView("ql_vs_click_web")

##print(process_start_date)
##print(process_end_date)
print(output_path)



In [0]:
## write base table to path, this will serve as a roll-up to weekly and monthly to reduce the processing time

bucket = "s3://analytics-scratch-reviewed-poshmark-production" ## 1yr bucket
## bucket = "s3://data-tmp-poshmark-production" ## 30 day bucket
team_name = "product_analytics"
project_name = "feed/visual_discovery/search"
data_name = "webviewDF"

output_path = f"{bucket}/{team_name}/{project_name}/{data_name}/"

look_sdf = spark.read.parquet(output_path)
look_sdf.createOrReplaceTempView("ql_vs_click_web")

##print(process_start_date)
##print(process_end_date)
print(output_path)


In [0]:
# finalDF = spark.sql(f"""
# with web as (select
#     event_date, 
#     on_name, 
#     collection_id,
#     page_group_id, 
#     user_id, 
#     unit_position, 
#     location, 
#     new_content_position, 
#     content_position, 
#     app_type, 
#     listing_id as listing_id, 
#     search_name, 
#     search_verb, 
#     search_unit_position, 
#     user_query, 
#     click_source,
#     count(user_id) as clicks
# from ql_vs_click_web
# group by 1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
# ) 
# , app as (select

#   event_date, 
#     on_name, 
#     collection_id,
#     page_group_id, 
#     user_id, 
#     unit_position, 
#     location, 
#     new_content_position, 
#     content_position, 
#     app_type, 
#     listing_id as listing_id, 
#     search_name, 
#     search_verb, 
#     search_unit_position, 
#     user_query, 
#     click_source,
#     count(user_id) as clicks
# from ql_vs_click_app
# group by 1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16

# )

# select * from app
# union all 
# select * from web
# """
# )


In [0]:
finalDF = spark.sql(f"""
select
  event_date, 
    on_name, 
    collection_id,
    page_group_id, 
    user_id, 
    unit_position, 
    location, 
    new_content_position, 
    content_position, 
    app_type, 
    listing_id as listing_id, 
    search_name, 
    search_verb, 
    search_unit_position, 
    user_query, 
    click_source,
    count(user_id) as clicks
from ql_vs_click_app
group by 1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
""")


s3_path_old = "s3://data-tmp-poshmark-production/product_analytics/feed/visual_discovery/search/finalDF/"
s3_path = "s3://analytics-scratch-reviewed-poshmark-production/product_analytics/feed/visual_discovery/search/finalDF/"
df = spark.read.format("parquet").load(s3_path_old)
df.repartition("event_date").write.partitionBy("event_date").format("parquet").mode("Overwrite").save(s3_path)


In [0]:

## write base table to path, this will serve as a roll-up to weekly and monthly to reduce the processing time

bucket = "s3://analytics-scratch-reviewed-poshmark-production" ## 1yr bucket
## bucket = "s3://data-tmp-poshmark-production" ## 30 day bucket
team_name = "product_analytics"
project_name = "feed/visual_discovery/search"
data_name = "finalDF"

output_path = f"{bucket}/{team_name}/{project_name}/{data_name}/"

## ## temp change to static from dynamic
##spark.conf.set("spark.sql.sources.partitionOverwriteMode","static")

finalDF.repartition("event_date").write.partitionBy("event_date").format("parquet").mode("Overwrite").save(output_path)
## base_orderitem.write.partitionBy("booked_date").format("parquet").mode("Overwrite").save(output_path)
## base_orderitem_df.write.format("parquet").mode("Overwrite").save(output_path)

## ## revert back to dynamic from static
## spark.conf.set("spark.sql.sources.partitionOverwriteMode","dynamic")

look_sdf = spark.read.parquet(output_path)
look_sdf.createOrReplaceTempView("ql_vs_click_final")

##print(process_start_date)
##print(process_end_date)
print(output_path)



In [0]:
bucket = "s3://analytics-scratch-reviewed-poshmark-production" ## 1yr bucket
##bucket = "s3://data-tmp-poshmark-production" ## 30 day bucket
team_name = "product_analytics"
project_name = "feed/visual_discovery/search"
data_name = "finalDF"

output_path = f"{bucket}/{team_name}/{project_name}/{data_name}/"
look_sdf = spark.read.parquet(output_path)
look_sdf.createOrReplaceTempView("ql_vs_click_final")

A user might interact with listing multiple times for multiple collection_id 

In [0]:

ordrDF = spark.sql(f"""
WITH RankedAttribution AS (
    select 
        a.event_date, 
        a.user_id,
        a.listing_id as listing_id,
        c.ttl as creator_id,
        c.pnm as pnm,
        a.collection_id, 
        a.click_source,
        app_type,
        case when ordr.is_valid_order = True then oi.order_id end as order_id,
        case when ordr.is_valid_order = True then oi.order_item_gmv_usd * 0.01 end order_item_gmv_usd,
        case when ordr.is_valid_order = True then oi.order_item_gmv_usd*0.01*0.055 end ltk_fee,
        case when ordr.is_valid_order = True then date(oi.booked_at_time) end ordr_date,
        CASE WHEN ordr.is_valid_order = True and ordr.offer_id IS NOT NULL THEN 'Y' ELSE 'N' END AS is_order_from_offer_yn,
        CASE WHEN ordr.is_valid_order = True and ordr.offer_id IS NOT NULL THEN DATE(offer.created_at) END AS offer_created_date,
        CASE WHEN ordr.is_valid_order = True and ordr.offer_id IS NOT NULL THEN offer.offer_initiated_by END AS offer_initiated_by,
        clicks, 
        -- PARTITION BY Item + Order to ensure 1 item only gets 1 click
        row_number() over(
            partition by 
                (CASE WHEN ordr.is_valid_order = TRUE THEN oi.order_id ELSE NULL END), 
                oi.listing_id 
            order by a.event_date desc
        ) as latest_click_rank
    from ql_vs_click_final a
    -- We join on user and listing to find possible interactions
    left join s3_analytics.dw_order_items oi 
        on a.listing_id = oi.listing_id 
        and a.user_id = oi.buyer_id
    left join s3_analytics.dw_orders ordr 
        on oi.order_id = ordr.order_id 
        and ordr.is_valid_order = TRUE
    left join s3_raw_mongo.looks c 
        on a.collection_id = c._id
    left join s3_analytics.dw_offers offer 
        on ordr.order_id = offer.order_id
)
select 
    event_date,
    user_id,
    listing_id,
    collection_id,
    click_source, 
    creator_id,
    pnm,
    app_type,
    CASE WHEN latest_click_rank = 1 THEN order_id END AS order_id,
    CASE WHEN latest_click_rank = 1 THEN order_item_gmv_usd END AS order_item_gmv_usd,
    CASE WHEN latest_click_rank = 1 THEN ltk_fee END AS ltk_fee,
    CASE WHEN latest_click_rank = 1 THEN ordr_date END AS ordr_date,
    CASE WHEN latest_click_rank = 1 THEN is_order_from_offer_yn END AS is_order_from_offer_yn,
    CASE WHEN latest_click_rank = 1 THEN offer_created_date END AS offer_created_date,
    CASE WHEN latest_click_rank = 1 THEN offer_initiated_by END AS offer_initiated_by,
    sum(clicks) as clicks_on_items
from RankedAttribution
group by 1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
; 
""")


In [0]:

## write base table to path, this will serve as a roll-up to weekly and monthly to reduce the processing time
bucket = "s3://analytics-scratch-reviewed-poshmark-production" ## 1yr bucket
#bucket = "s3://data-tmp-poshmark-production" ## 30 day bucket
team_name = "product_analytics"
project_name = "feed/visual_discovery/search"
data_name = "ordrDF"


output_path = f"{bucket}/{team_name}/{project_name}/{data_name}/"

## ## temp change to static from dynamic
##spark.conf.set("spark.sql.sources.partitionOverwriteMode","static")

ordrDF.repartition("event_date").write.partitionBy("event_date").format("parquet").mode("Overwrite").save(output_path)
## base_orderitem.write.partitionBy("booked_date").format("parquet").mode("Overwrite").save(output_path)
## base_orderitem_df.write.format("parquet").mode("Overwrite").save(output_path)

## ## revert back to dynamic from static
## spark.conf.set("spark.sql.sources.partitionOverwriteMode","dynamic")

look_sdf = spark.read.parquet(output_path)
look_sdf.createOrReplaceTempView("ql_vs_click_final_ordr")

##print(process_start_date)
##print(process_end_date)
print(output_path)



In [0]:

## write base table to path, this will serve as a roll-up to weekly and monthly to reduce the processing time
bucket = "s3://analytics-scratch-reviewed-poshmark-production" ## 1yr bucket
#bucket = "s3://data-tmp-poshmark-production" ## 30 day bucket
team_name = "product_analytics"
project_name = "feed/visual_discovery/search"
data_name = "ordrDF"

output_path = f"{bucket}/{team_name}/{project_name}/{data_name}/"

look_sdf = spark.read.parquet(output_path)
look_sdf.createOrReplaceTempView("ql_vs_click_final_ordr")

##print(process_start_date)
##print(process_end_date)




In [0]:
%sql 
select 
Click_source, 
--creator_id,
count(*),
count(distinct order_id), 
count(case when order_id is not null then listing_id end) oi
from ql_vs_click_final_ordr
where event_date between '2026-01-01' and '2026-01-31'
and creator_id is not null
and pnm = 'LTK'
group by 1
--order by 3 desc


### Sanity Check 
- all the order placed after event date 

In [0]:
reportDF = spark.sql(f"""
with clicks_items as (
select 
date(date_trunc('month', event_date)) as month_dt, 
creator_id, 
pnm, 
count(distinct collection_id) looks_cnt,
sum(clicks_on_items) clicks_on_items,
count(distinct listing_id) as listing_cnt,
count(distinct case when ordr_date = event_date or offer_created_date = event_date then order_id end ) order_id, 
sum(case when ordr_date = event_date or offer_created_date = event_date then order_item_gmv_usd end) gmv,  
sum(case when ordr_date = event_date or offer_created_date = event_date then ltk_fee end) ltk_fee
from ql_vs_click_final_ordr
where creator_id is not null --order_id is null or order_date = event_date or offer_created_date = event_date
group by 1,2,3
order by 8 desc
) 
,click_looks 
as (

select 
date(date_trunc('month', event_date)) as month_dt, 
creator_id, 
pnm, 
--count(distinct collection_id)collection2_cnt,
 sum(client_impressions) as client_impressions_on_look, 
 sum(clicks) as clicks_on_look
 --sum(client_impressions_users) as client_impressions_users,
--sum(clicks_users) as clicks_users
from visual_discovery_feed_creator_base
where collection_id is not null and sub_unit_type = 'look'
  group by 1,2,3
  order by 4 desc

)
select b.*, 
a.client_impressions_on_look,
a.clicks_on_look
 from click_looks a 
left join clicks_items b on a.creator_id = b.creator_id and a.month_dt = b.month_dt and a.pnm = b.pnm
order by ltk_fee desc
""")

In [0]:

## write base table to path, this will serve as a roll-up to weekly and monthly to reduce the processing time
bucket = "s3://analytics-scratch-reviewed-poshmark-production" ## 1yr bucket
## bucket = "s3://data-tmp-poshmark-production" ## 30 day bucket
team_name = "product_analytics"
project_name = "feed/visual_discovery/search"
data_name = "reportDF"

output_path = f"{bucket}/{team_name}/{project_name}/{data_name}/"

## ## temp change to static from dynamic
##spark.conf.set("spark.sql.sources.partitionOverwriteMode","static")

reportDF.repartition("month_dt").write.partitionBy("month_dt").format("parquet").mode("Overwrite").save(output_path)
## base_orderitem.write.partitionBy("booked_date").format("parquet").mode("Overwrite").save(output_path)
## base_orderitem_df.write.format("parquet").mode("Overwrite").save(output_path)

## ## revert back to dynamic from static
## spark.conf.set("spark.sql.sources.partitionOverwriteMode","dynamic")

look_sdf = spark.read.parquet(output_path)
look_sdf.createOrReplaceTempView("ql_vs_click_final_report")

##print(process_start_date)
##print(process_end_date)
print(output_path)



In [0]:

## write base table to path, this will serve as a roll-up to weekly and monthly to reduce the processing time
bucket = "s3://analytics-scratch-reviewed-poshmark-production" ## 1yr bucket
## bucket = "s3://data-tmp-poshmark-production" ## 30 day bucket
team_name = "product_analytics"
project_name = "feed/visual_discovery/search"
data_name = "reportDF"

output_path = f"{bucket}/{team_name}/{project_name}/{data_name}/"


look_sdf = spark.read.parquet(output_path)
look_sdf.createOrReplaceTempView("ql_vs_click_final_report")

##print(process_start_date)
##print(process_end_date)
print(output_path)



In [0]:
%sql 
select 
month_dt, 
sum(ltk_fee) ltk_fee 
from ql_vs_click_final_report
where 1=1 --creator_id is not null 
--and creator_id  not like "%Valentine's Look%"
--and creator_id not like "%Valentines Look%"
and month_dt between '2026-02-01' and '2026-03-01'
and pnm = 'LTK'
group by 1




In [0]:
%sql 
select 
month_dt, 
creator_id, 
sum(looks_cnt)looks_cnt, 
sum(clicks_on_items)clicks_on_items,
sum(clicks_on_look)clicks_on_look,
sum(order_id) order_cnt, 
sum(gmv) gmv,  
sum(ltk_fee) ltk_fee 
from ql_vs_click_final_report
where 1=1 --creator_id is not null 
--and creator_id  not like "%Valentine's Look%"
--and creator_id not like "%Valentines Look%"
and month_dt between '2026-02-01' and '2026-03-01'
and pnm = 'LTK'
group by 1,2
order by 8 desc



In [0]:
from pyspark.sql import *
from pyspark import *
from pyspark.sql import DataFrame

sc = spark._sc
helpers = sc._jvm.com.poshmark.spark.helpers
Config = helpers.Config
Redshift = helpers.Redshift
Config.reloadConfigs(sc._jsc.sc())
Config.registerUDFs()

In [0]:
%scala
      
// Imports & Inits
import com.poshmark.spark.helpers._
import org.apache.spark.sql.SaveMode

// Load default configs for helper libraries and register UDFs
Config.reloadConfigs(sc)
Config.registerUDFs()

////////////////////////////
//// Write to Redshift
// mode =SaveMode.Overwrite to back fill all data
//mode=SaveMode.Append to automate and add the process date 
var table_name = "analytics_scratch.ql_visual_discovery_ltk_report"
var sdf = spark.read.parquet("s3://analytics-scratch-reviewed-poshmark-production/product_analytics/feed/visual_discovery/search/reportDF/")
Redshift.writeDF(
  df=sdf,
   tableName=table_name,
   mode=SaveMode.Overwrite,
   postActions=s"Grant all on $table_name to group pm_users")